### Part II for Provider Master Prep -> Post Manual Tagging Fix

In [1]:
### Importing required libraries
import pandas as pd

In [2]:
### Reading the providers CSV file
providers_data = pd.read_csv('../data/bdc_us_provider_list_J25_03mar2026.csv')
print(providers_data.shape)
providers_data.head()


(2883, 3)


,frn,provider_id,holding_company
0,12609,240058,Mid-Hudson Cablevision
1,12781,240068,"Neu Ventures, Inc."
2,13722,131087,"Rainbow Telecommunications Association, Inc."
3,14936,240058,Mid-Hudson Cablevision
4,17640,131060,"TPC ACC Acquisition, LLC"


In [3]:
### Reading provider holding company XLSX sheet
providers_holding_company = pd.read_excel('../data/BDC Provider ID Table of Service Providers.xlsx')
print(providers_holding_company.shape)
providers_holding_company.head()

(4456, 5)


,Provider Name,Holding Company,Operation Type,FRN,Provider ID
0,"@Link Services, LLC","AtLink Services, LLC",Non-ILEC,16085920,290004
1,1 Point Communications,1 Point Communications,Non-ILEC,21352968,270002
2,101Netlink,101Netlink,Non-ILEC,18247254,190002
3,"123.Net, Inc","123.Net, Inc.",Non-ILEC,8590846,460000
4,"1Path Managed Services, LLC","1Path Managed Services, LLC",Non-ILEC,34347336,490000


In [4]:
### Renaming columns for merging
providers_data.rename(columns={'provider_id': 'Provider ID', 'frn': 'FRN'}, inplace=True)

In [5]:
### Merging the two datasets on the 'Provider ID' column
merged_data = pd.merge(providers_data, providers_holding_company, on=['Provider ID', 'FRN'], how='left')
### Displaying the merged dataset
print(merged_data.shape)
merged_data.head()

(2883, 6)


,FRN,Provider ID,holding_company,Provider Name,Holding Company,Operation Type
0,12609,240058,Mid-Hudson Cablevision,CATSKILL MOUNTAIN CABLEVISION,Mid-Hudson Cablevision,Non-ILEC
1,12781,240068,"Neu Ventures, Inc.","NEU VENTURES, INC.","Neu Ventures, Inc.",Non-ILEC
2,13722,131087,"Rainbow Telecommunications Association, Inc.",RAINBOW COMMUNICATIONS LLC,"Rainbow Telecommunications Association, Inc.",Non-ILEC
3,14936,240058,Mid-Hudson Cablevision,"MID-HUDSON CABLEVISION, Inc.",Mid-Hudson Cablevision,Non-ILEC
4,17640,131060,"TPC ACC Acquisition, LLC","Blue Stream Communications, LLC","TPC ACC Acquisition, LLC",Non-ILEC


##### As previously stated, the 'Holding Company Name' column contains many missing values, we will use the 'holding_company' column to fill these missing values

In [6]:
### Replacing missing 'Holding Company' values with 'holding_company' values
merged_data['Holding Company'] = merged_data['Holding Company'].fillna(merged_data['holding_company'])

In [7]:
### Obtaining mismatched rows between 'holding_company' and 'Holding Company' columns
mismatched_holding_company = merged_data[merged_data['holding_company'] != merged_data['Holding Company']]

### Obtaining rows with the same values for both columns
matched_holding_company = merged_data[merged_data['holding_company'] == merged_data['Holding Company']]

print("Mismatched Holding Company Shape:", mismatched_holding_company.shape)
print("Matched Holding Company Shape:", matched_holding_company.shape)


Mismatched Holding Company Shape: (63, 6)
Matched Holding Company Shape: (2820, 6)


##### Since all rows have values now, we will fix the discrepancies and create a master provider file for future analysis

In [8]:
### Importing the fixed manually tagged file to correct discrepancies
manually_tagged = pd.read_csv('../data/Mismatch Correction (Manually Tagged)  - mismatched_provider_holding_company.csv')
print(manually_tagged.shape)

(63, 7)


In [9]:
### Dropping the 'holding_company' column from the manually tagged dataset & renaming the 'Final Holding Company Name' column inplace to concatenate the values
manually_tagged.drop(columns=['holding_company'], inplace=True)
manually_tagged.rename(columns={'Final Holding Company Name': 'holding_company'}, inplace=True)
manually_tagged.head()

,FRN,Provider ID,Provider Name,Holding Company,Operation Type,holding_company
0,1584960,130029,Alaska Telephone Company,"Alaska Power & Telephone, Inc.",ILEC,Alaska Power & Telephone Company
1,1772888,130068,Ardmore Telephone Company Inc,"Synergy Technology Partners, inc.",ILEC,West Kentucky Rural Telephone Cooperative Corp...
2,1775295,131378,Brindlee Mountain Telephone LLC,Otelco Inc.,ILEC,"Future Fiber FinCo, LLC"
3,1792621,130068,West Kentucky Rural Telephone Cooperative Corp...,"Synergy Technology Partners, inc.",ILEC,West Kentucky Rural Telephone Cooperative Corp...
4,3007481,130619,The Chillicothe Telephone Co,"Horizon Telcom, Inc.",ILEC,Shenandoah Telecommunications Company


In [10]:
### Merging the two datasets to form the final master set
final_master_set = pd.concat([matched_holding_company, manually_tagged], ignore_index=True)
print("Final Master Set Shape:", final_master_set.shape)

Final Master Set Shape: (2883, 6)


In [11]:
### Choosing the required columns for the final master set
required_columns = ['Provider ID', 'FRN', 'holding_company']
final_master_set = final_master_set[required_columns]

### Saving the final master set to a CSV file
final_master_set.to_csv('../output/final_master_set.csv', index=False)